# MultiHop-RAG — Full Pipeline: Retrieval, Generation, RAGAS Evaluation


## 1. Install a compatible, pinned set of libraries
Same pinned stack as before, plus `faiss-cpu` for retrieval.

In [ ]:
!pip install -q \
  "ragas==0.2.15" \
  "langchain==0.3.27" "langchain-core==0.3.76" "langchain-community==0.3.30" \
  "langchain-openai==0.2.14" "langchain-groq==0.2.4" "langchain-huggingface==0.1.2" \
  datasets sentence-transformers faiss-cpu

print("Libraries installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

## 2. Add your API keys

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste Groq API key here: ")
print("Groq key stored for this session.")


Paste your Groq API key here: ··········
Groq key stored for this session.


In [ ]:
from huggingface_hub import login

login(getpass("Paste Hugging Face token here: "))


Paste your Hugging Face token here: ··········


## 3. Load both subsets and take a stratified sample

In [ ]:
from datasets import load_dataset
import pandas as pd

queries_dataset = load_dataset("yixuantt/MultiHopRAG", "MultiHopRAG")
corpus_dataset = load_dataset("yixuantt/MultiHopRAG", "corpus")

df_queries = queries_dataset["train"].to_pandas()
df_corpus = corpus_dataset["train"].to_pandas()

# Exclude null_query rows (no evidence, tests abstention, different question)
df_usable = df_queries[df_queries["question_type"] != "null_query"].reset_index(drop=True)

# Stratified sample: 5 rows per question type.
# Built with a simple loop + concat rather than groupby().apply() -- in some pandas
# versions, groupby().apply() silently drops the grouping column from the result,
# which would lose question_type entirely. This approach avoids that.
ROWS_PER_TYPE = 5
samples = []
for qtype in df_usable["question_type"].unique():
    subset = df_usable[df_usable["question_type"] == qtype]
    samples.append(subset.sample(n=ROWS_PER_TYPE, random_state=42))

df_sample = pd.concat(samples).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print(df_sample["question_type"].value_counts())


README.md:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

MultiHopRAG.json:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2556 [00:00<?, ? examples/s]

corpus.json:   0%|          | 0.00/6.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/609 [00:00<?, ? examples/s]

Sample shape: (15, 4)
question_type
inference_query     5
comparison_query    5
temporal_query      5
Name: count, dtype: int64


## 4. Build the retrieval index



In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_texts = df_corpus["body"].tolist()
corpus_embeddings = embed_model.encode(corpus_texts, show_progress_bar=True, convert_to_numpy=True)

# Normalize for cosine similarity via inner product
faiss.normalize_L2(corpus_embeddings)

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

print(f"Indexed {index.ntotal} articles.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Indexed 609 articles.


## 5. Retrieve the top-k most relevant articles per query

Using k=3, matching MultiHop-RAG's own design (evidence is drawn from 2-4 documents
per query, so 3 is a reasonable middle value).


In [ ]:
import textwrap

TOP_K = 3
MAX_CONTEXT_CHARS = 1200  # truncate each retrieved article to keep prompts within Groq's free-tier token budget
WRAP_WIDTH = 80

def retrieve_contexts(query, top_k=TOP_K):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, top_k)
    retrieved = []
    for idx in indices[0]:
        body = df_corpus.iloc[idx]["body"]
        retrieved.append(body[:MAX_CONTEXT_CHARS])
    return retrieved

df_sample["retrieved_contexts"] = df_sample["query"].apply(retrieve_contexts)

# Quick sanity check on one row
print("Example query:")
print(textwrap.fill(df_sample.iloc[0]["query"], WRAP_WIDTH))
print()
print("Retrieved contexts (truncated):")
for i, ctx in enumerate(df_sample.iloc[0]["retrieved_contexts"]):
    print(f"[{i}]")
    print(textwrap.fill(ctx[:200] + "...", WRAP_WIDTH))
    print()

Example query:
What country, featured in articles from both 'Fortune' and 'Business Today |
Latest Stock Market And Economy News India', has recently been involved in
issuing a relocation warning in Gaza, controlling the entry of essential
supplies there, and experienced a surprise attack due to an intelligence
failure?

Retrieved contexts (truncated):
[0]
Palestine’s growing tech industry has been literally blown apart by the war
between Israel and Hamas  Gaza, despite being one of the most economically
challenged regions in the world, has ironically a...

[1]
The world is still coming to terms with Hamas’ deadly attacks on Israelis last
weekend — and everything else that has unfolded so far in the aftermath,
including the barrage of retaliation strikes on ...

[2]
The world can ill-afford another full-fledged war, when economies are already
battling high inflation and interest rates, an ongoing Russia-Ukraine war and
now, a slowing China. This is precisely why ...



## 6. Generate an answer for each query using Groq


In [ ]:
from langchain_groq import ChatGroq
import time

generator_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

def generate_response(query, contexts):
    context_block = "\n\n".join(f"Source {i+1}: {c}" for i, c in enumerate(contexts))
    prompt = (
        f"Answer the question using only the information in the sources below. "
        f"Be concise and direct.\n\n{context_block}\n\nQuestion: {query}\nAnswer:"
    )
    result = generator_llm.invoke(prompt)
    return result.content

responses = []
for idx, row in df_sample.iterrows():
    print(f"Generating response {idx + 1}/{len(df_sample)}...")
    response = generate_response(row["query"], row["retrieved_contexts"])
    responses.append(response)
    time.sleep(2)  # small pause to stay well within Groq's free-tier rate limits

df_sample["response"] = responses
print()
print("Done. Example:")
print("Q:", df_sample.iloc[0]["query"])
print("A:", df_sample.iloc[0]["response"])
print("Gold answer:", df_sample.iloc[0]["answer"])


Generating response 1/15...
Generating response 2/15...
Generating response 3/15...
Generating response 4/15...
Generating response 5/15...
Generating response 6/15...
Generating response 7/15...
Generating response 8/15...
Generating response 9/15...
Generating response 10/15...
Generating response 11/15...
Generating response 12/15...
Generating response 13/15...
Generating response 14/15...
Generating response 15/15...

Done. Example:
Q: What country, featured in articles from both 'Fortune' and 'Business Today | Latest Stock Market And Economy News India', has recently been involved in issuing a relocation warning in Gaza, controlling the entry of essential supplies there, and experienced a surprise attack due to an intelligence failure?
A: The country is Israel.
Gold answer: Israel


## 7. Set up the LLM judge and run RAGAS

Same judge setup as `finqa` (Groq/Llama 3.1), but using the **standard, reference-based
Context Precision** this time, since a genuine gold `answer` has to be compared against.


In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.run_config import RunConfig

judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
ragas_llm = LangchainLLMWrapper(judge_llm)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

ragas_rows = []
for _, row in df_sample.iterrows():
    ragas_rows.append({
        "user_input": row["query"],
        "response": row["response"],
        "retrieved_contexts": row["retrieved_contexts"],
        "reference": row["answer"],
    })

eval_dataset = EvaluationDataset.from_list(ragas_rows)

slow_and_steady = RunConfig(
    timeout=300,
    max_workers=2,
    max_retries=15,
    max_wait=90,
)

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=slow_and_steady,
)

results_df = results.to_pandas()
results_df


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,"What country, featured in articles from both '...",[Palestine’s growing tech industry has been li...,The country is Israel.,Israel,0.250000,0.314526,0.583333
1,"Which company, recently reported by both TechC...",[A new class action lawsuit filed this week in...,Google.,Google,0.428571,0.312443,0.833333
2,"Which company, featured in multiple TechCrunch...","[ChatGPT, OpenAI’s viral AI chatbot, turns one...",The answer cannot be determined from the provi...,OpenAI,1.000000,-0.000000,1.000000
3,"Who, according to articles from TechCrunch, is...","[But eight years later, the argument between t...",There is not enough information in the provide...,Sam Altman,0.000000,0.000000,0.000000
4,"Which company, known for its e-reader lineup d...","[Table of Contents Table of Contents Echo, Fir...",Amazon.,Amazon,0.500000,0.369786,1.000000
5,"Does the TechCrunch article suggest that ""Peop...","[Save Log in , register or subscribe to save a...",There is no information in the provided source...,Yes,NaN,0.567660,0.333333
6,Does the TechCrunch article on GPT-4 suggest a...,[In a very swift test of the European Union’s ...,There is no information about a TechCrunch art...,Yes,1.000000,0.000000,1.000000
7,Does the TechCrunch article on Meta's GDPR com...,[Meta wants to shift the burden of monitoring ...,There is no TechCrunch article mentioned in th...,Yes,0.500000,0.423023,1.000000
8,Does the article from 'The Independent - Sport...,[Sign up to Miguel Delaney’s Reading the Game ...,"No, the article from 'The Independent - Sports...",Yes,0.666667,0.531473,0.000000
9,Does the TechCrunch article discussing Meta's ...,[Meta wants to shift the burden of monitoring ...,"No, the TechCrunch articles do not mention Pal...",Yes,0.200000,0.332808,1.000000


## 8. Check for and retry any failed rows

Same approach as `finqa` — free-tier timeouts are normal, not a broken pipeline.

In [ ]:
metric_cols = ["faithfulness", "answer_relevancy", "context_precision"]

missing_counts = results_df[metric_cols].isna().sum()
print("Missing values per metric:")
print(missing_counts)


Missing values per metric:
faithfulness         1
answer_relevancy     0
context_precision    0
dtype: int64


In [ ]:
failed_mask = results_df[metric_cols].isna().any(axis=1)
failed_indices = results_df[failed_mask].index.tolist()

print(f"Retrying {len(failed_indices)} row(s): {failed_indices}")

if len(failed_indices) > 0:
    retry_rows = [ragas_rows[i] for i in failed_indices]
    retry_dataset = EvaluationDataset.from_list(retry_rows)

    retry_results = evaluate(
        dataset=retry_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=slow_and_steady,
    )
    retry_df = retry_results.to_pandas()

    for pos, orig_idx in enumerate(failed_indices):
        for col in metric_cols:
            if pd.isna(results_df.loc[orig_idx, col]):
                results_df.loc[orig_idx, col] = retry_df.loc[pos, col]

    print("Retry complete. Remaining missing values:")
    print(results_df[metric_cols].isna().sum())
else:
    print("No missing values - nothing to retry.")


Retrying 1 row(s): [5]


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Retry complete. Remaining missing values:
faithfulness         0
answer_relevancy     0
context_precision    0
dtype: int64


## 9. Build the final results table and save it

In [ ]:
comparison_df = pd.concat([
    df_sample[["query", "question_type", "answer", "response"]].reset_index(drop=True),
    results_df[metric_cols].reset_index(drop=True)
], axis=1)

comparison_df.to_csv("multihop_rag_final_results.csv", index=False)
print("Saved: multihop_rag_final_results.csv")
comparison_df


Saved: multihop_rag_final_results.csv


,query,question_type,answer,response,faithfulness,answer_relevancy,context_precision
0,"What country, featured in articles from both '...",inference_query,Israel,The country is Israel.,0.250000,0.314526,0.583333
1,"Which company, recently reported by both TechC...",inference_query,Google,Google.,0.428571,0.312443,0.833333
2,"Which company, featured in multiple TechCrunch...",inference_query,OpenAI,The answer cannot be determined from the provi...,1.000000,-0.000000,1.000000
3,"Who, according to articles from TechCrunch, is...",inference_query,Sam Altman,There is not enough information in the provide...,0.000000,0.000000,0.000000
4,"Which company, known for its e-reader lineup d...",inference_query,Amazon,Amazon.,0.500000,0.369786,1.000000
5,"Does the TechCrunch article suggest that ""Peop...",comparison_query,Yes,There is no information in the provided source...,0.833333,0.567660,0.333333
6,Does the TechCrunch article on GPT-4 suggest a...,comparison_query,Yes,There is no information about a TechCrunch art...,1.000000,0.000000,1.000000
7,Does the TechCrunch article on Meta's GDPR com...,comparison_query,Yes,There is no TechCrunch article mentioned in th...,0.500000,0.423023,1.000000
8,Does the article from 'The Independent - Sport...,comparison_query,Yes,"No, the article from 'The Independent - Sports...",0.666667,0.531473,0.000000
9,Does the TechCrunch article discussing Meta's ...,comparison_query,Yes,"No, the TechCrunch articles do not mention Pal...",0.200000,0.332808,1.000000


## 10. Compare average scores by (native) question type

Using MultiHop-RAG's own ground-truth question types, not a heuristic tagger.


In [ ]:
print("Average scores by question type:")
print(comparison_df.groupby("question_type")[metric_cols].mean().round(3))
print()
print("Row counts per group:")
print(comparison_df["question_type"].value_counts())


Average scores by question type:
                  faithfulness  answer_relevancy  context_precision
question_type                                                      
comparison_query         0.640             0.371              0.667
inference_query          0.436             0.199              0.683
temporal_query           0.550             0.453              0.467

Row counts per group:
question_type
inference_query     5
comparison_query    5
temporal_query      5
Name: count, dtype: int64


In [ ]:
import pandas as pd
import textwrap

pd.set_option('display.max_colwidth', 60)

df = pd.read_csv("multihop_rag_final_results.csv")

WRAP_WIDTH = 80

# ---------- TABLE 3: terse-but-correct answers ----------
print("=" * WRAP_WIDTH)
print("Terse-but-correct answers")
print("=" * WRAP_WIDTH)

table3_targets = ["Google.", "Amazon.", "The country is Israel."]
table3 = df[df["response"].isin(table3_targets)][
    ["response", "answer", "answer_relevancy"]
].reset_index(drop=True)
table3.columns = ["Response", "Gold Answer", "Answer Relevancy"]
table3["Answer Relevancy"] = table3["Answer Relevancy"].round(3)

print(table3.to_string(index=False))
print()

Terse-but-correct answers
              Response Gold Answer  Answer Relevancy
The country is Israel.      Israel             0.315
               Google.      Google             0.312
               Amazon.      Amazon             0.370



In [ ]:

pd.set_option('display.max_colwidth', 60)
df = pd.read_csv("multihop_rag_final_results.csv")

WRAP_WIDTH = 80

print("=" * WRAP_WIDTH)
print("Confidently wrong answer scoring perfectly")
print("=" * WRAP_WIDTH)

table4_row = df[df["response"].str.contains("GPT-4", na=False)].iloc[0]

print("QUERY:")
print(textwrap.fill(table4_row["query"], WRAP_WIDTH))
print()
print("RESPONSE:")
print(textwrap.fill(table4_row["response"], WRAP_WIDTH))
print()

table4_summary = pd.DataFrame([{
    "Gold Answer": table4_row["answer"],
    "Faithfulness": round(table4_row["faithfulness"], 3),
    "Context Precision": round(table4_row["context_precision"], 3),
}])
print(table4_summary.to_string(index=False))

Confidently wrong answer scoring perfectly
QUERY:
Does the TechCrunch article on GPT-4 suggest a different level of susceptibility
to producing toxic text compared to other large language models, while the
TechCrunch article on the European Commission's probe into Elon Musk’s X focuses
on concerns about illegal content and disinformation in a different context?

RESPONSE:
There is no information about a TechCrunch article on GPT-4 in the provided
sources. However, the sources do discuss concerns about disinformation and
illegal content on X (formerly Twitter) and the European Union's efforts to
address these issues.

Gold Answer  Faithfulness  Context Precision
        Yes           1.0                1.0


## Progress Note — RAGAS Metric Recomputation on MultiHop-RAG

Built a retrieval-augmented pipeline from scratch for MultiHop-RAG, since (unlike RAGBench) it provides no pre-generated response: embedded all 609 corpus articles using sentence-transformers, built a FAISS index, retrieved the top-3 most relevant articles per query, and generated an answer for each using Groq (Llama 3.1). Sample was stratified (5 rows each) across MultiHop-RAG's three usable native question types — comparison_query, inference_query, temporal_query — with null_query (301 rows, all lacking supporting evidence) set aside as a separate abstention-testing case rather than mixed into the main analysis. Used RAGAS's standard, reference-based Context Precision this time, since MultiHop-RAG (unlike finqa) provides a genuine gold answer.

**Result**: Successfully scored all 15 rows across all three metrics (2 initial timeouts resolved via retry).